In [0]:
from pyspark.sql import functions as F

# Percentages documented per requirement #9
DUPLICATE_FRACTION = 0.005     # 0.5%
LATE_EVENT_FRACTION = 0.003    
INVALID_FARE_FRACTION = 0.002  
MISSING_ID_FRACTION = 0.002   
INVALID_TS_FRACTION = 0.001   
ANOMALY_FRACTION = 0.002      
UNKNOWN_ZONE_FRACTION = 0.0005

def inject_duplicates(df, fraction=DUPLICATE_FRACTION, seed=101):
    dup_sample = df.sample(fraction=fraction, seed=seed)
    return df.unionByName(dup_sample)

def inject_late_events(df, fraction=LATE_EVENT_FRACTION, seed=103):
    return df.withColumn(
        "event_timestamp",
        F.when(F.rand(seed=seed) < fraction,
               F.col("event_timestamp") - F.expr("INTERVAL 6 HOURS"))
         .otherwise(F.col("event_timestamp"))
    )

def inject_invalid_fares(df, fraction=INVALID_FARE_FRACTION, seed=105):
    return df.withColumn(
        "fare_amount",
        F.when((F.rand(seed=seed) < fraction) & F.col("fare_amount").isNotNull(),
               F.col("fare_amount") * -1)
         .otherwise(F.col("fare_amount"))
    )

def inject_missing_ids(df, fraction=MISSING_ID_FRACTION, seed=107):
    return df.withColumn(
        "driver_id",
        F.when((F.rand(seed=seed) < fraction) & F.col("driver_id").isNotNull(),
               F.lit(None).cast("string"))
         .otherwise(F.col("driver_id"))
    )

def inject_invalid_timestamps(df, fraction=INVALID_TS_FRACTION, seed=109):
    return df.withColumn(
        "dropoff_datetime",
        F.when(F.rand(seed=seed) < fraction,
               F.col("pickup_datetime") - F.expr("INTERVAL 1 HOUR"))
         .otherwise(F.col("dropoff_datetime"))
    )

def inject_anomalies(df, fraction=ANOMALY_FRACTION, seed=111):
    is_anomaly = F.rand(seed=seed) < fraction
    is_speed_type = F.rand(seed=seed + 1) < 0.5
    return (
        df
        .withColumn("fare_amount",
            F.when(is_anomaly & ~is_speed_type & F.col("fare_amount").isNotNull(),
                   F.col("fare_amount") * 50)
             .otherwise(F.col("fare_amount")))
        .withColumn("distance_km",
            F.when(is_anomaly & is_speed_type & F.col("distance_km").isNotNull(),
                   F.col("distance_km") * 25)
             .otherwise(F.col("distance_km")))
    )

def inject_unknown_zones(df, fraction=UNKNOWN_ZONE_FRACTION, seed=113):
    return df.withColumn(
        "pickup_zone_id",
        F.when(F.rand(seed=seed) < fraction, F.lit(9999))
         .otherwise(F.col("pickup_zone_id"))
    )

def apply_all_bad_data(df):
    df = inject_duplicates(df)
    df = inject_late_events(df)
    df = inject_invalid_fares(df)
    df = inject_missing_ids(df)
    df = inject_invalid_timestamps(df)
    df = inject_anomalies(df)
    df = inject_unknown_zones(df)
    return df